# Notebook to analyse sourcephotonly_w_miri fits!

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import pandas as pd
import os

import scipy
import numpy as np
import astropy
import scipy.stats as stats

from astropy.table import Table
from prospector_utils.analysis import analyse_miri_fits
from prospector_utils.plotting import *

quiescent = [7549, 8013, 8469, 9395, 10128, 10339, 10400, 10565, 10592, 11142, 11494, 16419, 18668, 21477]
no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
#below_ms = [10600, 18977, 21451]
bl_agn = [12020, 18977]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = no_spec + intermediate_ap + bl_agn

print(f"I should exclude {len(exclude)} galaxies from my analysis with Prospector.")

In [ ]:
table_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_flux_scaled.fits'

table = Table.read(table_path, format='fits')
galaxy_ids = np.asarray([str(gid) for gid in table['ID']])

ids_to_reconstruct = [gid for gid in galaxy_ids if int(gid) not in no_spec and int(gid) not in bl_agn]
print(f"I will reconstruct the SEDs of {len(ids_to_reconstruct)} galaxies with Prospector.")

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/fits/'
appphot_only_wMIRI = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI'
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/pickle_files/'
phot_table = '/Users/benjamincollins/University/Master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_flux_scaled.fits'

In [ ]:
analyse_miri_fits(phot_table=phot_table,
                  data_dir=appphot_only_wMIRI,
                  plot_dir=plot_dir,
                  stats_dir=stats_dir,
                  add_dust=True)

# Suppressing dust emission in 17517

In [ ]:
import prospect.io.read_results as reader
from prospector_utils.params import get_MAP
from prospect.sources import FastStepBasis
from prospect.utils.plotting import posterior_samples

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/fits_nodust/'
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/pickle_files_nodust/'
phot_table = '/Users/benjamincollins/University/Master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

# Load the h5 file for the given objid
data_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.5/appphot_only_wMIRI_cat/'

objid = 17517

h5_file = glob.glob(os.path.join(data_dir, f"output_{objid}*.h5"))

try:
    h5_file = h5_file[0]
except IndexError:
    print(f"No PROSPECTOR results found for objid {objid}.")

# Load PROSPECTOR results
full_path = os.path.join(data_dir, h5_file)
results, obs, model = reader.results_from(full_path)

filename = os.path.join(stats_dir, f"{objid}.pkl")

#print(obs['filters'])
#return

# Now we have to exclude the last 3 parameters from the fit
map_parameters = get_MAP(results)

#print(map_parameters)

# Build the MAP dictionary
MAP = {}
for a,b in zip(results['theta_labels'], map_parameters):
    MAP[a] = b
    
zred = MAP['zred']
logmass = MAP['logmass']
dust2 = MAP['dust2']    # extract the diffuse dust V-band optical depth

# Calculate the spectrum based on the Maximum A Posteriori (MAP) parameters
sps = FastStepBasis(zcontinuous=1)

# 1. Create a copy of your MAP parameters
no_dust_params = map_parameters.copy()

# 2. Toggle the model setting to False
# This prevents the code from adding the IR 'glow'
model.params['add_dust_emission'] = np.array([False])

# 3. Predict the spectrum
# The resulting spec/phot will show the attenuated UV but NO IR emission
spec, phot, _ = model.predict(no_dust_params, obs=obs, sps=sps)    

# Convert maggies to µJy
maggies_to_muJy = 3631e6

# Wavelengths of the model spectrum
wave_spec = sps.wavelengths

# Convert to arrays
phot = np.array(phot)

# Draw 100 posterior samples
samples = posterior_samples(results, 100)

sample_specs = []
for params_i in samples:    
    spec_i, _, _ = model.predict(params_i, obs=obs, sps=sps)
    sample_specs.append(spec_i)
sample_specs = np.array(sample_specs)  # shape: (nsample, nwave)

# Takes the per-pixel percentiles such that the final spectra are not actual spectra of Prospectors parameter space
lower = np.percentile(sample_specs, 16, axis=0)
median = np.percentile(sample_specs, 50, axis=0)
upper = np.percentile(sample_specs, 84, axis=0)

# Compute filter wavelength in microns
phot_wave = np.array([filt.wave_effective for filt in obs['filters']])  # in Angstroms

data = {
    # Important metadata
    'id': objid,
    'zred': zred,
    'maggies_to_muJy': maggies_to_muJy,
    
    'model': {
        'spec_best': spec,
        'spec_16th': lower,
        'spec_median': median,
        'spec_84th': upper,
        'wave_spec': wave_spec,
        'sample_specs': sample_specs[:10],
        'phot': phot,
        'phot_wave': phot_wave
    },
    
    # One entry for the observation dictionary
    'obs': obs,
    
    #'fit_quality': fit_quality,
    
    'galaxy_properties': {
        'logmass': logmass,
        'dust2': dust2
    },
    
    # Moved it outside so it's easier accessible
    'map_theta': MAP # Keep the full raw dictionary just in case    
}

# Write output to a pickle file
with open(filename, 'wb') as f:
    pkl.dump(data, f)
print(f"💾 Saved data to {filename}")
    
if plot_dir:
    plot_miri_fit(filename, plot_dir)


Plot fit with COSMOS2025 Data

In [ ]:
plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/fits_nodust/'
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/pickle_files_nodust/'

objid = 17517

filename = os.path.join(stats_dir, f"{objid}.pkl")

plot_miri_fit(filename, plot_dir)

# Compare MIRI parameters with no MIRI

In [ ]:
filename = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files/17517.pkl'

with open(filename, 'rb') as f:
    fit_data = pkl.load(f)

print(fit_data)

Restructure pickle files to have map_parameters easily accessible:

In [ ]:
stats_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files'

output_folder = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/pickle_files_new'

os.makedirs(output_folder, exist_ok=True)

file_paths = glob.glob(os.path.join(stats_dir, "*.pkl"))
print(f"Found {len(file_paths)} files. Starting restructuring...")

for f_path in file_paths:
    with open(f_path, 'rb') as f:
        old_data = pkl.load(f)

    # 1. Extract raw MAP dictionary
    gal_props = old_data.get('galaxy_properties', {})
    
    MAP = gal_props.get('map_theta', {})
    
    # 2. Extract observational data
    # If obs_miri isn't a separate key yet, we take it from 'obs'
    obs_data = old_data.get('obs', {})

    # 3. Reconstruct Galaxy Properties
    # We pull these from the existing map_theta
    galaxy_props = {
        'logmass': gal_props.get('logmass'),
        'dust2': gal_props.get('dust2'),
        'sfr_100myr': gal_props.get('sfr_100myr'), # Call helper for math
        'sfr_30myr': gal_props.get('srf_30myr'), 
        'sfr_bins': gal_props.get('sfr_bins'), # Add your specific bin array if available
        'agebins': gal_props.get('agebins')   # Add your specific agebins array if available
    }

    # 4. Create New Structure
    new_data = {
        'id': old_data.get('id'),
        'zred': old_data.get('zred'),
        'maggies_to_muJy': old_data.get('maggies_to_muJy'),
        'model': old_data.get('model'),
        'obs': obs_data,
        'fit_quality': old_data.get('fit_quality', {}), # Carry over if exists
        'galaxy_properties': galaxy_props,
        'map_theta': MAP # Now outside and easily accessible
    }

    # 5. Save the new file
    new_filename = os.path.join(output_folder, os.path.basename(f_path))
    with open(new_filename, 'wb') as f:
        pkl.dump(new_data, f)

print("Restructuring complete.")

# Plot NSigma and R distributions

Prepare the data

In [ ]:
pickle_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/pickle_files/'
plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/fit_quality/'
os.makedirs(plot_dir, exist_ok=True)

pickle_files = glob.glob(f'{pickle_dir}/*.pkl')

all_data = []
reduced_chi2_list = []

# 1. DATA EXTRACTION
for filename in pickle_files:
    with open(filename, 'rb') as f:
        data = pkl.load(f)
        
    gid = data['id']
    
    if gid in exclude:
        continue
    
    fit_quality = data.get('fit_quality', {})
    
    # Store global per-galaxy stats
    if 'chi2_red' in fit_quality:
        reduced_chi2_list.append({
            'galaxy_id': gid,
            'reduced_chi2': fit_quality['chi2_red'],
            'n_filters': len(fit_quality) - 1 # Assuming only 'chi2_red' is non-filter
        })

    # Store per-band stats
    for i, (key, val) in enumerate(fit_quality.items()):
        if isinstance(val, dict): # This identifies the filter entries
            all_data.append({
                'galaxy_id': gid,
                'filter_name': key,
                'N_sigma': val.get('n_sigma') * (-1),
                'flux': val.get('obs_flux'),
                'flux_err': val.get('obs_err'),
                'flux_mod': val.get('mod_flux'),
                'flux_mod_err': val.get('mod_err'),
                'frac_diff': val.get('frac_diff') * (-1),
                'snr': val.get('snr')
            })
            
            if val.get('n_sigma') > 7.0:
                print(gid, key)
                print(np.log10(val.get('obs_flux')))
                print(np.log10(val.get('mod_flux')))
                print("Ratio:", np.log10(val.get('obs_flux')/val.get('mod_flux')))
                print("Nsigma:", val.get('n_sigma'))
                print("\n")

df = pd.DataFrame(all_data)
chi2_df = pd.DataFrame(reduced_chi2_list)


bands = ['F770W', 'F1000W', 'F1800W', 'F2100W']
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']  # Distinct colors per band

Plot N Sigma distributions

In [ ]:
# PLOT 1: N_SIGMA HISTOGRAMS
#fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=False, sharey=True)
fig, axes = plt.subplots(2, 2, figsize=(8, 6.5), sharex=False, sharey=True)
axes = axes.flatten()  # easier to index

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    nsigmas = subset['N_sigma']
    if len(nsigmas) == 0: continue
    
    ax.set_title(f'{band}')
    #ax.set_xlim(x_min, x_max)
    ax.set_xlabel(r'$\mathrm{N_\sigma}$', fontsize=12)
    ax.set_ylabel('Number of galaxies', fontsize=12)
    
    #if i in [0,1]: ax.set_ylim(0, 24)
    #elif i in [2,3]: ax.set_ylim(0,12)

    # Add compact statistics
    mean_ratio = np.mean(nsigmas)
    median_ratio = np.median(nsigmas)
    std_ratio = np.std(nsigmas)
    mad_ratio = median_abs_deviation(nsigmas)
    N = len(subset['galaxy_id'].unique())
    num = f'N = {N}'
    
    x_min = -8.5
    x_max = 8.5
    bins = np.linspace(x_min, x_max, 25)
    
    counts, bin_edges, _ = ax.hist(nsigmas, bins=bins, color=colors[i], alpha=0.7, edgecolor='black')

    x = np.linspace(x_min, x_max, 500)
    gaussian_norm = stats.norm.pdf(x, loc=0, scale=1)
    gaussian_obs = stats.norm.pdf(x, loc=median_ratio, scale=mad_ratio)

    # Scale Gaussians to match histogram counts
    gaussian_norm_scaled = gaussian_norm * len(nsigmas) * (bin_edges[1] - bin_edges[0])
    gaussian_obs_scaled = gaussian_obs * len(nsigmas) * (bin_edges[1] - bin_edges[0])
    
    ax.plot(x, gaussian_norm_scaled, 'gray', lw=2, alpha=1, label=r'$\mathcal{N}(0,1)$')
    ax.plot(x, gaussian_obs_scaled, colors[i], lw=2, alpha=1, label=r'$\mathcal{N}_{\mathrm{obs}}$')
    
    ax.vlines(0.0, ymin=0, ymax=25, color='black', alpha=0.6, linestyle='--', linewidth=2)
    ax.vlines(median_ratio, ymin=0, ymax=25, color=colors[i], alpha=0.6, linestyle='--', linewidth=2)
    
    median_ratio = np.median(nsigmas)
    
    stats_text = f'Med = {median_ratio:.2f}\nMAD = {mad_ratio:.2f}\n{num}'
    ax.legend(loc="upper right")
    #ax.text(0.78, 0.84, stats_text, transform=ax.transAxes, fontsize=10,
    #        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax.text(0.03, 0.76, stats_text, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    # Annotate in the top-right corner (adjust x,y if needed)
    #ax.text(0.95, 0.95, f'N = {n_galaxies}', 
    #        transform=ax.transAxes, ha='right', va='top',
    #        fontsize=10, bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
    
#plt.suptitle(r'$N_\sigma$ distribution for each MIRI filter', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
# Save single combined figure
filename = os.path.join(plot_dir, 'Nsigma_gauss.png')
plt.savefig(filename, dpi=300)
plt.show()

Plot the log-ratios

In [ ]:
# PLOT 2: LOG RATIOS
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)
axes = axes.flatten()  # easier to index

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    
    f_obs = subset['flux']
    err = subset['flux_err']
    frac_diff = subset['frac_diff']
    
    # SNR filter: Keep only detections > 3-sigma
    snr_mask = (f_obs / err) >= 3.0
    
    # Model reconstruction
    # Using the simplified: f_model = f_obs * (1 - frac_diff)
    # We filter out frac_diff >= 1 to avoid log(0) or log(negative)
    valid_mask = (frac_diff < 1.0) & snr_mask
    
    # Calculate ratios only for valid entries
    # log10(f_obs / (f_obs * (1-frac_diff))) reduces to -log10(1-frac_diff)
    log_ratios = -np.log10(1.0 - frac_diff[valid_mask])
    
    ax.set_title(f'{band}')
    #ax.set_xlim(x_min, x_max)
    ax.set_xlabel('Flux ratio (dex)')
    ax.set_ylabel('Number of galaxies')
    
    #if i in [0,1]: ax.set_ylim(0, 24)
    #elif i in [2,3]: ax.set_ylim(0,12)
    
    # Add compact statistics
    mean_logr = np.mean(log_ratios)
    std_logr = np.std(log_ratios)
    median_logr = np.median(log_ratios)
    mad_logr = median_abs_deviation(log_ratios)
    N = len(log_ratios)
    num = f'N = {N}'
    
    x_min = -1.2
    x_max = 1.2
    bins = np.linspace(x_min, x_max, 25)
    
    counts, _, _ = ax.hist(log_ratios, bins=bins, color=colors[i], alpha=0.7, edgecolor='black')

    ymax = np.max(counts) * 1.1 # for all plots
    ymax = max(ymax, 10)
    ax.set_ylim(0,ymax)
    ax.vlines(median_logr, ymin=0, ymax=ymax, color='darkred', alpha=0.8, linestyle='-', linewidth=2, label=f'Median: {median_logr:.2f}')
#            if i == 2: 
#               stats_text += ' (*)'
#              print(log_ratios[log_ratios > 1])
    ax.text(0.025, 0.92, num, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.plot([],[], label=f'MAD: {mad_logr:.2f}', alpha=0)  # dummy plot for legend
    ax.legend()
    
#plt.suptitle(r'$N_\sigma$ distribution for each MIRI filter', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
# Save single combined figure
filename = os.path.join(plot_dir, 'log_ratios.png')
plt.savefig(filename, dpi=300)
plt.show()

Plot the reduced Chi squared

In [ ]:
# Compute reduced chi^2 per galaxy

chi2_red = chi2_df['reduced_chi2']

# --- 1. Clipping & Statistics ---
chi2_red = chi2_df['reduced_chi2']
q95 = chi2_red.quantile(0.95)
filtered = chi2_df[chi2_red <= q95]

mean_val = chi2_red.mean()
median_val = chi2_red.median()
mad_val = median_abs_deviation(chi2_red.dropna())

fig, axes = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={"width_ratios":[1.25,0.75]})

# --- Left: Nsigma vs sSFR
counts, bins, patches = axes[0].hist(filtered['reduced_chi2'], bins=25, alpha=0.7, edgecolor='black', range=(0, q95))

axes[0].set_xlabel(r'Reduced $\chi^2$')
axes[0].set_ylabel('Number of galaxies')

# Count how many chi2 values are in the histogram
chi2_values = len(chi2_df)
num = f'\nN = {len(filtered)}/{chi2_values}\n(95th perctile)'
# Annotate in the top-right corner (adjust x,y if needed)

# plot vertical lines for mean and median
# Plot vertical lines for mean and median
ymax = counts.max() * 1.1
axes[0].vlines(mean_val, ymin=0, ymax=ymax, color='red', alpha=0.8, linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
axes[0].vlines(median_val, ymin=0, ymax=ymax, color='darkred', alpha=0.8, linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')
axes[0].plot([],[], label=num, alpha=0)  # dummy plot for legend
axes[0].set_ylim(0, ymax)
axes[0].legend()    


# Scatter plot with filtered data
#plt.scatter(filtered['n_filters'], filtered['reduced_chi2'], alpha=0.7)    
axes[1].scatter(filtered['n_filters'], filtered['reduced_chi2'], alpha=0.7)    
axes[1].set_xlabel('Number of photometric data points')
axes[1].set_ylabel(r'Reduced $\chi^2$')
#plt.title(r'Reduced $\chi^2$ vs. number of MIRI bands')
axes[1].axhline(1, color='orange', linestyle='--', label='Unity')
axes[1].legend()    

plt.tight_layout()
filename = os.path.join(plot_dir, 'reduced_chi2.png')
plt.savefig(filename, dpi=300)
plt.show()

threshold = 30  # user-specified value
high_chi2_ids = chi2_df.loc[chi2_df['reduced_chi2'] > threshold, 'galaxy_id'].tolist()
print(f"✅ Saved reduced chi^2 plots to {filename}")
print(f"{len(high_chi2_ids)} galaxies have reduced χ² > {threshold}")
print("These galaxies are:", high_chi2_ids)
print("Their χ² values are:", chi2_df.loc[chi2_df['reduced_chi2'] > threshold, 'reduced_chi2'].tolist())

Now one plot showing the flux ratios of model and error

In [ ]:
# PLOT: LOG FLUX MODEL vs LOG FLUX OBS (1 row, 4 panels)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=False, sharey=False)

for i, (ax, band) in enumerate(zip(axes, bands)):
    subset = df[df['filter_name'] == band]
    f_obs = subset['flux']
    err_obs = subset['flux_err']
    f_mod = subset['flux_mod']
    err_mod = subset['flux_mod_err']
    frac_diff = subset['frac_diff']
    snr = subset['snr']

    # SNR filter + require both fluxes to be positive
    valid_mask = (snr >= 3.0) & (f_obs > 0) & (f_mod > 0) & (err_obs > 0) & (err_mod > 0)

    f_obs_v = f_obs[valid_mask].values
    f_mod_v = f_mod[valid_mask].values
    
    # CORRECT - propagated uncertainty in log space
    err_obs_v = err_obs[valid_mask].values
    err_mod_v = err_mod[valid_mask].values
    
    N = valid_mask.sum()

    ax.errorbar(f_obs_v, f_mod_v, 
                    xerr=err_obs_v, yerr=err_mod_v,
                    fmt='o', alpha=0.7, capsize=2, markersize=6, 
                    color=colors[i], label=f'N = {N}')
    
    ax.loglog()  # switches to log scale with physical tick values

    # 1:1 line in flux space
    lims_low  = min(f_obs_v.min(), f_mod_v.min()) * 0.6
    lims_high = max(f_obs_v.max(), f_mod_v.max()) * 1.4
    ax.plot([lims_low, lims_high], [lims_low, lims_high], 'k--', linewidth=1, label='1:1')
    ax.set_xlim(lims_low, lims_high)
    ax.set_ylim(lims_low, lims_high)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=14)
    

    ax.set_title(band, fontsize=20)
    ax.set_xlabel(r'$f_\mathrm{obs}$ [µJy]', fontsize=18)
    if i == 0:
        ax.set_ylabel(r'$f_\mathrm{model}$ [µJy]', fontsize=18)

    ax.legend(fontsize=13, loc='upper left')
    #ax.grid(True, alpha=0.3)

plt.tight_layout()
filename = os.path.join(plot_dir, 'log_flux_model_vs_obs.png')
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()

# Comparison between the galaxy parameters with and without MIRI!

In [ ]:
with_miri = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/pickle_files/'
no_miri = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/sourcephotonly/flux_scaled/pickle_files/'

plot_dir = '/Users/benjamincollins/University/master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled/parameter_comparisons/'
os.makedirs(plot_dir, exist_ok=True)

def extract_summary_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Extract the scalar properties we want to compare
            # We flatten the 'galaxy_properties' nested dict here
            entry = {
                'id': data['id'],
                'zred': data['zred'],
                **data['galaxy_properties'] # Unpacks logmass, dust2, sfr_100myr, etc.
            }
            
            if data['id'] in exclude:
                print("Skipping galaxy...")
                continue
            
            summary_list.append(entry)
            
    return pd.DataFrame(summary_list)

# 1. Define your paths (update these to your actual folder/naming convention)
path_no_miri = glob.glob(os.path.join(no_miri,'*.pkl'))
path_with_miri = glob.glob(os.path.join(with_miri,'*.pkl'))

# 2. Extract into DataFrames
df_no_miri = extract_summary_data(path_no_miri)
df_with_miri = extract_summary_data(path_with_miri)

# 3. Merge the two datasets on 'id'
# We add suffixes so we can tell which logmass is which
df_comparison = pd.merge(
    df_no_miri, 
    df_with_miri, 
    on='id', 
    suffixes=('_nomiri', '_miri')
)

print(f"Successfully merged data for {len(df_comparison)} galaxies.")

Plot the dust parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle as pkl
import glob
import numpy as np

def calculate_statistics(x, y, valid_mask):
    """
    Calculate comparison statistics
    """
    if np.sum(valid_mask) < 3:
        return {}
    
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    
    # Linear correlation
    corr_coef, p_value = stats.pearsonr(x_valid, y_valid)
    
    # Calculate residuals and statistics
    residuals = y_valid - x_valid
    mean_residual = np.mean(residuals)
    median_residual = np.median(residuals)
    std_residual = np.std(residuals)
    rms_residual = np.sqrt(np.mean(residuals**2))
    
    # Fractional differences for positive values
    frac_diff = (y_valid - x_valid) / x_valid
    median_frac_diff = np.median(frac_diff)
    mean_frac_diff = np.mean(frac_diff)
    std_frac_diff = np.std(frac_diff)
    
    return {
        'correlation': corr_coef,
        'p_value': p_value,
        'median_residual': median_residual,
        'mean_residual': mean_residual,
        'std_residual': std_residual,
        'rms_residual': rms_residual,
        'median_frac_diff': median_frac_diff,
        'mean_frac_diff': mean_frac_diff,
        'std_frac_diff': std_frac_diff,
        'n_objects': len(x_valid)
    }

def extract_dust_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Helper function to find parameter in either dict
            def get_param(name):
                if name in data['galaxy_properties']:
                    return data['galaxy_properties'][name]
                elif name in data['map_theta']:
                    return data['map_theta'][name]
                return np.nan

            entry = {
                'id': data['id'],
                'dust1_fraction': get_param('dust1_fraction') * get_param('dust2'),
                'dust_index': get_param('dust_index'),
                'dust2': get_param('dust2')
            }
            
            if data['id'] in exclude:
                continue
            
            summary_list.append(entry)
    return pd.DataFrame(summary_list)

# 1. Update these paths to your actual folders
path_no_miri = glob.glob(os.path.join(no_miri,'*.pkl'))
path_with_miri = glob.glob(os.path.join(with_miri,'*.pkl'))

# 2. Extract and Merge
df_no = extract_dust_data(path_no_miri)
df_with = extract_dust_data(path_with_miri)
df = pd.merge(df_no, df_with, on='id', suffixes=('_no', '_with'))

# 3. Create the 3x1 Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
dust_params = [
    ('dust1_fraction', 'dust1', '#4C72B0'),
    ('dust_index', 'dust_index', '#55A868'),
    ('dust2', 'dust2', '#C44E52')
]

for i, (col, label, colour) in enumerate(dust_params):
    ax = axes[i]
    
    x = df[f'{col}_with']
    y = df[f'{col}_no']
    
    corr_coef, p_value = stats.pearsonr(x, y)
    
    ax.scatter(x, y, alpha=0.6, edgecolors='black', s=45, color=colour)
    
    if i==1: 
        ax.set_xlim(-0.75, 0.75)
        ax.set_ylim(-0.75, 0.75)
    
    # Force square aspect ratio so 1-to-1 actually looks like a 45-degree line
    ax.set_aspect('equal', adjustable='box')
    
    plt.draw() 
    ticks = ax.get_xticks()
    ax.set_yticks(ticks)
    print(ticks)
    
    #if i in [0, 1]:
    if i == 1:
        x_min = ticks[0] + 0.25
        x_max = ticks[-1]
    
        ax.set_xticks(np.arange(x_min, x_max, 0.5))
        ax.set_yticks(np.arange(x_min, x_max, 0.5))
    
        #ax.set_xticks(np.arange(-1, 1, 0.5))
        #ax.set_yticks(np.arange(-1, 1, 0.5))
    
    else:
        x_min = ticks[1]
        x_max = 5#ticks[-1] + 1
        
        ax.set_xticks(np.arange(x_min, x_max, 1))
        ax.set_yticks(np.arange(x_min, x_max, 1))
        ax.set_xlim(-0.2)
        ax.set_ylim(-0.2)
            
    # Add a 1-to-1 reference line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
    
    
    ax.set_title(f'{label}', fontsize=15)
    ax.set_xlabel('With MIRI', fontsize=13)
    ax.set_ylabel('No MIRI', fontsize=13)
    ax.tick_params(labelsize=12)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    ax.text(0.05, 0.95, f'r = {corr_coef:.3f}',
                transform=ax.transAxes, verticalalignment='top', fontsize=12,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))


#plt.suptitle("Dust Absorption Parameters", fontsize=17)
filename = os.path.join(plot_dir, 'dust_params_v2.png')

plt.tight_layout()
plt.savefig(filename, dpi=300)
print(f"Figure saved to {filename}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle as pkl
import glob
import numpy as np

def extract_dust_data(file_list):
    summary_list = []
    for f in file_list:
        with open(f, 'rb') as p:
            data = pkl.load(p)
            
            # Helper function to find parameter in either dict
            def get_param(name):
                if name in data['galaxy_properties']:
                    return data['galaxy_properties'][name]
                elif name in data['map_theta']:
                    return data['map_theta'][name]
                return np.nan

            entry = {
                'id': data['id'],
                'duste_umin': get_param('duste_umin'),
                'duste_qpah': get_param('duste_qpah'),
                'duste_gamma': get_param('duste_gamma')
            }
            
            if data['id'] in exclude:
                continue
            
            summary_list.append(entry)
    return pd.DataFrame(summary_list)

# 1. Update these paths to your actual folders
path_no_miri = glob.glob(os.path.join(no_miri,'*.pkl'))
path_with_miri = glob.glob(os.path.join(with_miri,'*.pkl'))

# 2. Extract and Merge
df_no = extract_dust_data(path_no_miri)
df_with = extract_dust_data(path_with_miri)
df = pd.merge(df_no, df_with, on='id', suffixes=('_no', '_with'))

# 3. Create the 3x1 Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
dust_params = [
    ('duste_umin', r'$U_{min}$ ', '#4C72B0'),
    ('duste_qpah', 'PAH-fraction', '#55A868'),
    ('duste_gamma', r'Dust $\gamma$', '#C44E52')
]

for i, (col, label, colour) in enumerate(dust_params):
    ax = axes[i]
    
    x = df[f'{col}_with']
    y = df[f'{col}_no']
    
    ax.scatter(x, y, alpha=0.6, edgecolors='black', s=60, color=colour)
    
    # Add a 1-to-1 reference line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
    
    ax.set_title(f'{label}', fontsize=15)
    ax.set_xlabel('With MIRI', fontsize=13)
    ax.set_ylabel('No MIRI', fontsize=13)
    ax.tick_params(labelsize=12)
    #ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle("Dust Emission Parameters", fontsize=17)
filename = os.path.join(plot_dir, 'duste_params.png')

plt.tight_layout()
plt.savefig(filename, dpi=300)
print(f"Figure saved to {filename}")
plt.show()